# The Duckiedrone (DD24-B) Flight Control Architecture

The fundamental flight control operations, such as achiving a certain attitude or translating high level flight commands to low level PWM signals to each motors on a drone are performed by the flight controller hardware component. This dedicated "low level brain" is critical due to the high bandwidth operations of a quadcopter, meaning that commands to the motors must be provided at high frequencies (typically >200Hz) to maintain the drone on a stable flight configuration and not crash.

Wrapped around the lower-level (or _inner_) control loop we find a slower (typically around 20 Hz) higher-level (or _outer_) control loop that provides the setpoints to the lower level loop. This means that there are several (PID) control loops operating at the same time on a drone at any time, defining the "flight control architecture". 

To understand how the Duckiedrone works, we start by considering the flight controller and its firmware. 

## The flight controller: PX4

The Duckiedrone (**DD24-B** model) uses an open-source firmware for the flight controller called **PX4**. PX4 handles the low-level attitude control (roll, pitch, yaw) internally, running at 1 kHz, and exposes a high-level interface to companion computers via **MAVLink**. We talk to PX4 over ROS2 Jazzy using **MAVROS2**, a bridge that translates between ROS2 messages and MAVLink.

## PX4 OFFBOARD Mode

PX4 supports several flight modes (Position, Altitude, Stabilized, etc.). For companion-computer control we use **OFFBOARD** mode, which lets an external computer stream setpoints directly to the flight controller.

**Key rules for OFFBOARD mode:**

1. **Heartbeat requirement**: PX4 must receive setpoints at **> 2 Hz**, continuously. If this stream of setpoints stops for more than ~0.5s, PX4 will exit OFFBOARD mode and apply a failsafe action (e.g., Return-to-Land).
2. **Pre-arm streaming**: Setpoints must be streaming *before* OFFBOARD mode is activated. The node on the Duckiedrone streams 100 idle setpoints before initializing OFFBOARD mode.
3. **Manual override**: An RC transmitter mode switch always overrides OFFBOARD; this is your safety net.

## The `setpoint_attitude` MAVROS2 Plugin

For altitude control we use the MAVROS2 [`setpoint_attitude`](https://wiki.ros.org/mavros#mavros.2FPlugins.setpoint_attitude) plugin, which maps to PX4's `VehicleAttitudeSetpoint` uORB message.

It exposes two independent topics that must be published together:

| Topic | Message type | Description |
|-------|-------------|-------------|
| `/mavros/setpoint_attitude/attitude` | `geometry_msgs/PoseStamped` | Desired orientation as a quaternion |
| `/mavros/setpoint_attitude/thrust`   | `mavros_msgs/Thrust` | Normalized thrust in **[0.0, 1.0]** |

> **Note on thrust conventions**: MAVROS2 uses a **[0, 1]** normalized range, where 0 is no thrust and 1 is full thrust. PX4's internal `VehicleAttitudeSetpoint` uses **[-1, 1]** for the body-z axis; MAVROS2 handles the conversion automatically.

## Altitude Control PID Loop in practice

Our altitude control PID works at the *outer* loop level:

```
┌─────────────┐     error      ┌──────────────────┐    thrust [0,1]
│  Setpoint z ├───────────────►│  Altitude PID    ├──────────────────►┐
└─────────────┘                │  (your code!)    │                   │
                               └──────────────────┘                   │
                                       ▲                              │
                                       │ altitude (m)                 │
                               ┌───────┴──────────┐                   │
                               │  Bottom range    │                   │
                               │  finder (ToF)    │                   │
                               └──────────────────┘                   │
                                                                       ▼
                      level quaternion ──────►  MAVROS2  ──► PX4 (inner loop)
                                            setpoint_attitude    roll/pitch/yaw
                                                                   + thrust
                                                                       │
                                                                       ▼
                                                                    Motors
```

**Process Variable**: altitude above ground (meters), measured by the bottom time-of-flight rangefinder.

**Setpoint**: desired altitude (meters).

**Error**: `e = setpoint_z − current_z` (positive when drone is below target).

**Control Variable**: normalized thrust between [0.0, 1.0].

## The Hover Thrust Offset $K$

To make the Duckiedrone fly (as opposed to fall), the motors will have to continuosly generate a lift force that at least counteracts the effect of gravity, i.e., a force (equal or) greater than the weight of the quadcopter. Since the weight of the Duckiedrone, in our application, does not change over time, we can separate this constant component of the generated thrust and call it **hover thrust offset**: the normalized thrust at which the drone neither climbs nor descends:

$$K = \frac{m g}{F_{\max}}$$

where $F_{\max}$ is the maximum thrust the motors can produce. For the DD24-B, $K$ is typically around **0.44**.

The PID control action is then:

$$u(t_k) = K_p e(t_k) + K_i \sum_{i=0}^{k} e(t_i)\,\Delta t + K_d \frac{e(t_k) - e(t_{k-1})}{\Delta t} + K$$

And the output $u$ is clipped to **[0.0, 1.0]** at every time instant $$t_k$$.

## Safety Architecture

The `pid_controller_node` implements a three-state safety machine:

| State | Behaviour |
|-------|-----------|
| `PREFLIGHT` | Streams 100 idle setpoints (thrust = 0.1, level attitude) to satisfy the OFFBOARD pre-arm requirement. |
| `OFFBOARD_IDLE` | OFFBOARD mode is active and the vehicle is armed. Idle thrust (0.1) keeps the heartbeat alive without flying. |
| `ACTIVE` | After calling `/enable_altitude_control`, the PID computes real thrust. |

The **`/enable_altitude_control`** ROS2 service (type `std_srvs/Trigger`) acts as a **safety gate** which must explicitly be called after verifying that the drone is ready to fly. Calling it again disables the PID (toggles back to `OFFBOARD_IDLE`).

If the range sensor stops publishing for more than 1 second while in `ACTIVE` state, the node automatically reverts to `OFFBOARD_IDLE`.

## Joystick Attitude Override

For manual planar positioning (moving the drone left/right/forward/backward), one can publish a `geometry_msgs/PoseStamped` to `/dd24/attitude_override`. The node will use that quaternion instead of the default level attitude for the next 0.5s, allowing to manually nudge the drone's position while the altitude PID maintains height.

# Thinking Activities

Let's reflect on the above.

## Duckiedrone altitude controller 

The PID control equation for the Duckiedrone's altitude controller is:

$$u(t_k) = K_p e(t_k) + K_i \sum_{i=0}^{k} e(t_i)\,\Delta t + K_d \frac{e(t_k) - e(t_{k-1})}{\Delta t} + K$$

where $u \in [0.0, 1.0]$ is the normalized thrust sent to PX4 via MAVROS2.

### Altitude Control
Suppose you are implementing an altitude PID controller for the DD24 (vertical movement only).

1. What is the **process variable**, the **error**, and the **control variable** for this altitude PID controller?

2. What is $K$ in the context of the DD24?  What happens physically if $K$ is set **too high** (e.g. 0.9)?  What if $K$ is set **too low** (e.g. 0.1)?

3. Why is the output clipped to [0.0, 1.0]?  What physical constraint does this represent?

### OFFBOARD Mode

4. Why must setpoints be published at **> 2 Hz** when using PX4 OFFBOARD mode?  What happens if the stream stops?

5. Explain the role of the **safety gate** (`/enable_altitude_control` service) in the node.  Why is it necessary to explicitly enable the PID rather than having it run automatically from startup?

### Handin

Write your answers to the questions above in `answers_pid.md` inside the `packages/solution/solution/` folder.